# Model Packaging and Reproducibility

## Objective

This notebook packages the final heart disease prediction model into a reusable
scikit-learn pipeline.

The notebook will:

1. Load the prepared heart disease dataset.
2. Recreate the final preprocessing and model pipeline.
3. Train the selected final model.
4. Save the complete pipeline using Joblib.
5. Reload the saved pipeline.
6. Verify that predictions remain identical after loading.
7. Demonstrate prediction on a single patient record.

Saving preprocessing and the model together ensures that training and inference
use exactly the same data transformations.

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# Define Project path

In [2]:
PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "heart_cleaned.csv"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_PATH = MODEL_DIR / "heart_disease_pipeline.joblib"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)
print("Model output path:", MODEL_PATH)

Project root: d:\MLOps\heart-disease-mlops
Dataset path: d:\MLOps\heart-disease-mlops\data\processed\heart_cleaned.csv
Model output path: d:\MLOps\heart-disease-mlops\models\heart_disease_pipeline.joblib


## Load the Cleaned Dataset

The cleaned dataset created during exploratory data analysis is loaded from the
`data/processed` directory.

Before training the final pipeline, the dataset shape, columns, data types, and
missing values are checked to confirm that the correct data is being used.

In [3]:
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

df.head()

Dataset loaded successfully.
Dataset shape: (303, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,1
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0


# Inspect columns and missing values

In [4]:
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values per column:")
print(df.isnull().sum())

Column names:
['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']

Data types:
age           int64
sex           int64
cp            int64
trestbps      int64
chol          int64
fbs           int64
restecg       int64
thalach       int64
exang         int64
oldpeak     float64
slope         int64
ca          float64
thal        float64
target        int64
dtype: object

Missing values per column:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64


## Separate Features and Target

The `target` column is the value that the model must predict.

All remaining columns are input features describing the patient's health
characteristics.

In [5]:
TARGET_COLUMN = "target"

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)
print("\nTarget distribution:")
print(y.value_counts().sort_index())

Feature matrix shape: (303, 13)
Target vector shape: (303,)

Target distribution:
target
0    164
1    139
Name: count, dtype: int64


## Define Numerical and Categorical Features

The dataset contains two broad feature types:

- Numerical measurements, such as age, cholesterol, resting blood pressure, and
  maximum heart rate.
- Categorical or discrete medical indicators, such as sex, chest pain type,
  fasting blood sugar, and exercise-induced angina.

The feature groups are explicitly recorded so that preprocessing remains
consistent and reproducible.

In [6]:
numerical_features = [
    "age",
    "trestbps",
    "chol",
    "thalach",
    "oldpeak",
]

categorical_features = [
    "sex",
    "cp",
    "fbs",
    "restecg",
    "exang",
    "slope",
    "ca",
    "thal",
]

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)
print("Total features:", len(numerical_features) + len(categorical_features))

Numerical features: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
Categorical features: ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
Total features: 13


# Validate the expected columns

In [7]:
expected_features = numerical_features + categorical_features
actual_features = X.columns.tolist()

missing_features = sorted(set(expected_features) - set(actual_features))
unexpected_features = sorted(set(actual_features) - set(expected_features))

print("Missing expected features:", missing_features)
print("Unexpected features:", unexpected_features)

assert not missing_features, f"Missing features: {missing_features}"
assert not unexpected_features, f"Unexpected features: {unexpected_features}"

print("\nFeature validation passed.")

Missing expected features: []
Unexpected features: []

Feature validation passed.


## Build the Preprocessing Pipeline

The preprocessing pipeline ensures that every future dataset is transformed in
exactly the same way as the training data.

Packaging preprocessing together with the trained model guarantees
reproducibility and prevents inconsistencies between training and inference.

In [8]:
# Numerical preprocessing pipeline
numerical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

# Categorical preprocessing pipeline
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
    ]
)

# Combine both preprocessing pipelines
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

print(preprocessor)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['age', 'trestbps', 'chol', 'thalach',
                                  'oldpeak']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent'))]),
                                 ['sex', 'cp', 'fbs', 'restecg', 'exang',
                                  'slope', 'ca', 'thal'])])


## Split the Dataset

The cleaned dataset is divided into training and testing sets.

The training set is used to fit the complete pipeline, while the testing set is
kept separate for evaluating the packaged model.

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Training feature shape :", X_train.shape)
print("Testing feature shape  :", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts().sort_index())

print("\nTesting target distribution:")
print(y_test.value_counts().sort_index())

Training feature shape : (242, 13)
Testing feature shape  : (61, 13)

Training target distribution:
target
0    131
1    111
Name: count, dtype: int64

Testing target distribution:
target
0    33
1    28
Name: count, dtype: int64


## Build the Final Machine Learning Pipeline

The preprocessing pipeline and the final Random Forest classifier are combined
into one reusable scikit-learn Pipeline.

Packaging preprocessing together with the model ensures that future predictions
use exactly the same transformations as were used during training.

In [10]:
# Create the complete machine learning pipeline
model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=100,
                random_state=42,
            ),
        ),
    ]
)

print(model_pipeline)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'trestbps', 'chol',
                                                   'thalach', 'oldpeak']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent'))]),
                                                  ['sex', 'cp', 'fbs',
                                                   'restecg', 'exang', 'slope',
               

## Train the Final Pipeline

The complete machine learning pipeline is trained using the training dataset.

During training, the preprocessing steps are fitted first, followed by the
Random Forest classifier.

In [11]:
# Train the complete pipeline
model_pipeline.fit(X_train, y_train)

print("Pipeline training completed successfully.")

Pipeline training completed successfully.


## Evaluate the Packaged Pipeline

The trained pipeline is evaluated on the unseen test dataset.

The evaluation confirms that the packaged preprocessing and classifier work
together correctly before the pipeline is saved.

In [12]:
# Generate class predictions
y_pred = model_pipeline.predict(X_test)

# Generate probability estimates for the positive class
y_pred_proba = model_pipeline.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics
test_accuracy = accuracy_score(y_test, y_pred)
test_roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test ROC-AUC : {test_roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))

Test Accuracy: 0.9180
Test ROC-AUC : 0.9540

Confusion Matrix:
[[29  4]
 [ 1 27]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9667    0.8788    0.9206        33
           1     0.8710    0.9643    0.9153        28

    accuracy                         0.9180        61
   macro avg     0.9188    0.9215    0.9179        61
weighted avg     0.9227    0.9180    0.9182        61



The final Random Forest pipeline achieved a test accuracy of 91.80% and a ROC-AUC score of 0.9540, demonstrating excellent predictive performance on unseen patient data. The confusion matrix showed only one false negative, indicating that the model successfully identified the majority of patients with heart disease. The high recall (96.43%) for the positive class makes the model particularly suitable for medical screening applications, where minimizing missed disease cases is important.

## Save the Trained Pipeline

The complete machine learning pipeline, including preprocessing and the trained
Random Forest classifier, is saved using Joblib.

Saving the entire pipeline ensures that future predictions use the same
preprocessing steps as the training process, improving reproducibility.

In [13]:
# Save the complete pipeline
joblib.dump(model_pipeline, MODEL_PATH)

print("Pipeline saved successfully.")
print("Saved model path:", MODEL_PATH)

Pipeline saved successfully.
Saved model path: d:\MLOps\heart-disease-mlops\models\heart_disease_pipeline.joblib


# Reload the Saved Pipeline

In [14]:
# Load the saved pipeline
loaded_pipeline = joblib.load(MODEL_PATH)

print("Pipeline loaded successfully.")
print(type(loaded_pipeline))

Pipeline loaded successfully.
<class 'sklearn.pipeline.Pipeline'>


# Verify Reproducibility

In [15]:
# Predictions using the loaded pipeline
loaded_predictions = loaded_pipeline.predict(X_test)

# Compare predictions from the original and loaded pipelines
predictions_match = np.array_equal(y_pred, loaded_predictions)

print("Predictions identical:", predictions_match)

assert predictions_match, "Loaded model predictions do not match the original model."

print("Reproducibility check passed successfully.")

Predictions identical: True
Reproducibility check passed successfully.


# Example Inference Using the Saved Pipeline

The saved pipeline is used to predict the heart disease status of a single
patient record.

This demonstrates how the packaged model can be used during deployment without
performing manual preprocessing.

In [16]:
# Select one patient from the test dataset
sample_patient = X_test.iloc[[0]]

# Actual diagnosis
actual_label = y_test.iloc[0]

# Prediction
predicted_label = loaded_pipeline.predict(sample_patient)[0]

# Prediction probability
prediction_probability = loaded_pipeline.predict_proba(sample_patient)[0]

print("Sample patient:")
display(sample_patient)

print(f"\nActual label      : {actual_label}")
print(f"Predicted label   : {predicted_label}")
print(f"Probability (No Disease): {prediction_probability[0]:.4f}")
print(f"Probability (Disease)   : {prediction_probability[1]:.4f}")

Sample patient:


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
219,59,1,4,138,271,0,2,182,0,0.0,1,0.0,3.0



Actual label      : 0
Predicted label   : 0
Probability (No Disease): 0.7500
Probability (Disease)   : 0.2500


# Conclusion

In this notebook:

- A reusable preprocessing pipeline was created.
- The preprocessing pipeline and Random Forest classifier were combined into a
  single scikit-learn Pipeline.
- The complete pipeline was trained and evaluated.
- The trained pipeline was saved using Joblib.
- The saved pipeline was reloaded successfully.
- Predictions from the original and loaded pipelines were verified to be
  identical, demonstrating reproducibility.
- A sample inference was performed using the packaged pipeline.

The packaged model is now ready to be integrated into a REST API for deployment.